# Fine-tune YOLOv8 for Volleyball Ball Detection (ROUND 2)

**Before starting:**
1. Run `python scripts/prepare_dataset_for_training.py datasets/ball_detection/` locally
2. Upload the generated `ball_dataset.zip` to this Colab
3. Upload `volleyball_ball_best.pt` (from the project's `models/`) to this Colab
   — round 2 fine-tunes FROM the production weights (see section 3)
4. Select **Runtime > Change runtime type > T4 GPU**
5. Run all cells

## 1. Setup

In [ ]:
!pip install ultralytics -q

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Upload and Extract Dataset

Upload `ball_dataset.zip` using the file browser (folder icon in the left sidebar),
then run the cell below.

In [ ]:
import zipfile
from pathlib import Path

# Upload the zip first using the Colab file browser (left sidebar)
# Or uncomment below to use an upload dialog:
# from google.colab import files
# uploaded = files.upload()  # select ball_dataset.zip

zip_path = "/content/ball_dataset.zip"
extract_dir = "/content/dataset"

zipfile.ZipFile(zip_path).extractall(extract_dir)

DATASET_YAML = f"{extract_dir}/data.yaml"
print(f"Extracted to: {extract_dir}")

In [ ]:
# Verify dataset structure
!echo "=== data.yaml ==="
!cat /content/dataset/data.yaml
!echo ""
!echo "=== Train images ==="
!ls /content/dataset/train/images | wc -l
!echo "=== Valid images ==="
!ls /content/dataset/valid/images | wc -l

## 3. Fine-tune

**ROUND 2: start FROM the production weights** (`volleyball_ball_best.pt`,
uploaded to /content). Round 1 (v2_match) started from base COCO
`yolov8n.pt` and LOST the sky-backed confidence prior on unseen episodes
(control-ep35 med conf 0.90 → 0.47) while confident FPs exploded
(330 → 684). The round-2 dataset adds 200 sand-noise negatives + 150 sky
controls to attack exactly that; starting from best.pt preserves the sky
prior. Set `BASE_MODEL = "yolov8n.pt"` to reproduce round 1.

Key settings (unchanged):
- **imgsz=1280**: ball is tiny (~20-40px), needs high resolution input
- **freeze=10**: keep backbone features, only retrain detection head
- **patience=15**: early stopping if validation loss plateaus

In [ ]:
from ultralytics import YOLO

BASE_MODEL = "/content/volleyball_ball_best.pt"  # ROUND 2; "yolov8n.pt" reproduces round 1
model = YOLO(BASE_MODEL)

results = model.train(
    data=DATASET_YAML,
    epochs=100,
    imgsz=1280,
    batch=8,
    freeze=10,
    patience=15,
    device=0,
    project="/content/runs",
    name="volleyball_ball_r2",
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.4,
    mosaic=0.5,
    scale=0.3,
    flipud=0.0,
    fliplr=0.5,
)

## 4. Evaluate

In [ ]:
metrics = model.val()

print(f"\nmAP50:     {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

In [ ]:
from IPython.display import Image, display
display(Image(filename="/content/runs/volleyball_ball_r2/results.png", width=800))

In [ ]:
display(Image(filename="/content/runs/volleyball_ball_r2/val_batch0_pred.jpg", width=800))

## 5. Download Weights

Download `volleyball_ball_best_r2.pt` and place it in the project root
(the validation-gate session stashes it under `models/` — production
`volleyball_ball_best.pt` must stay untouched for the A/B).

Then use it:
```python
ball_detector = BallDetector(model_path="models/volleyball_ball_best_r2.pt")
```

In [ ]:
import shutil

best_weights = Path("/content/runs/volleyball_ball_r2/weights/best.pt")
download_path = Path("/content/volleyball_ball_best_r2.pt")
shutil.copy(best_weights, download_path)

print(f"Size: {download_path.stat().st_size / 1024 / 1024:.1f} MB")

from google.colab import files
files.download(str(download_path))